In [ ]:
import os, json, random, math, time, csv
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm
# ============================ USER PATHS ============================
IMG_DIR   = r"/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/images"
MASK_DIR  = r"/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/masks"
SAVE_PATH = r"/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/micromodel_v3_compressed_train.keras"  # must end with .keras

USE_MIXED_PRECISION = False
FAST_STEM  = True
IMG_SIZE   = 160                      # [COMPRESS] resolution compression (256 -> 160); must be divisible by 32
CHANNELS   = (32, 64, 96, 128, 192)
EPOCHS     = 100                      # [COMPRESS] fewer epochs; the faster pipeline converges quickly
BATCH_SIZE = 8                        # increased from 4
MAX_LR     = 3e-3
WEIGHT_DECAY = 1e-4
PATIENCE   = 20
SEED       = 42
N_XAI_IMAGES = 8
USE_CACHE  = True                     # [COMPRESS] reuse compressed .npz cache between runs

TARGETS = dict(gmacs=1.0, params_m=0.5, size_mb=2.0, metric=0.95, fps=50.0)

assert SAVE_PATH.endswith(".keras"), "SAVE_PATH must end with .keras"
assert IMG_SIZE % (32 if FAST_STEM else 16) == 0, "IMG_SIZE must be divisible by 32 (FAST_STEM) or 16"
BASE = SAVE_PATH[:-len(".keras")]
STEM_STRIDE = 2 if FAST_STEM else 1
UP_MODE = "nearest" if USE_MIXED_PRECISION else "bilinear"
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
AUTOTUNE = tf.data.AUTOTUNE
METRIC_KEYS = ["dice", "iou", "precision", "recall", "specificity", "accuracy"]
def seed_everything(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)


# ------------------------------- MODEL -------------------------------
def dsconv(x, cout, name, last=False):
    x = layers.SeparableConv2D(cout, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU(name=f"{name}_out" if last else None)(x)


def block(x, cout, name):
    x = dsconv(x, cout, name)
    return dsconv(x, cout, name, last=True)

def build_model(ch=CHANNELS, size=IMG_SIZE):
    c0, c1, c2, c3, c4 = ch
    inp = layers.Input((size, size, 3))
    x = layers.Conv2D(c0, 3, strides=STEM_STRIDE, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(name="stem_out")(x)

    s0 = block(x, c0, "enc1")
    s1 = block(layers.MaxPool2D()(s0), c1, "enc2")
    s2 = block(layers.MaxPool2D()(s1), c2, "enc3")
    s3 = block(layers.MaxPool2D()(s2), c3, "enc4")
    b  = block(layers.MaxPool2D()(s3), c4, "bott")

    up = lambda t: layers.UpSampling2D(2, interpolation=UP_MODE)(t)
    d3 = block(layers.Concatenate()([up(b),  s3]), c3, "dec4")
    d2 = block(layers.Concatenate()([up(d3), s2]), c2, "dec3")
    d1 = block(layers.Concatenate()([up(d2), s1]), c1, "dec2")
    d0 = block(layers.Concatenate()([up(d1), s0]), c0, "dec1")

    if FAST_STEM:
        out = layers.Conv2D(1, 1, dtype="float32", name="head")(d0)
        out = layers.UpSampling2D(2, interpolation=UP_MODE, dtype="float32", name="logits")(out)
    else:
        out = layers.Conv2D(1, 1, dtype="float32", name="logits")(d0)
    return Model(inp, out, name="LiteUNet")

def count_params(m):
    return m.count_params() / 1e6

def count_gmacs(model):
    macs = 0
    for l in model.layers:
        if isinstance(l, layers.SeparableConv2D):
            h, w, cout = l.output.shape[1:4]; cin = l.input.shape[-1]; k = l.kernel_size[0]
            macs += h * w * cin * k * k + h * w * cin * cout
        elif isinstance(l, layers.Conv2D):
            h, w, cout = l.output.shape[1:4]; cin = l.input.shape[-1]; k = l.kernel_size[0]
            macs += h * w * cout * cin * k * k
    return macs / 1e9

# ------------------------------- DATA --------------------------------
def build_splits(img_dir, mask_dir, seed=SEED):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    imgs  = sorted(f for f in os.listdir(img_dir) if f.lower().endswith(exts))
    masks = {os.path.splitext(f)[0]: f for f in os.listdir(mask_dir) if f.lower().endswith(exts)}
    pairs = [(os.path.join(img_dir, f), os.path.join(mask_dir, masks[os.path.splitext(f)[0]]))
             for f in imgs if os.path.splitext(f)[0] in masks]
    assert pairs, "No image/mask pairs found - check paths and filenames."
    rng = random.Random(seed); rng.shuffle(pairs)
    n = len(pairs); n_tr = int(0.8 * n); n_va = int(0.1 * n)
    return pairs[:n_tr], pairs[n_tr:n_tr + n_va], pairs[n_tr + n_va:]

def load_pair(ip, mp):
    img = cv2.imread(ip); msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    assert img is not None, f"Cannot read image: {ip}"
    assert msk is not None, f"Cannot read mask: {mp}"
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # INTER_AREA is the best interpolation for shrinking (keeps detail when compressing resolution)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return img, (msk > 127).astype(np.uint8)

# [COMPRESS] ---------------------------------------------------------
def compress_dataset(pairs, name):
    """Data compression step. Decodes + downsizes images ONCE, keeps them as uint8
    (4x smaller than float32), and stores them in a compressed .npz cache.
    Later runs load the cache in seconds instead of re-decoding every JPEG."""
    cache = f"{BASE}_cache_{name}_{IMG_SIZE}_{len(pairs)}.npz"
    if USE_CACHE and os.path.exists(cache):
        d = np.load(cache)
        print(f"[{name}] loaded compressed cache: {os.path.basename(cache)}")
        return d["imgs"], d["msks"]
    data = [load_pair(ip, mp) for ip, mp in tqdm(pairs, desc=f"compress {name}", leave=False)]
    imgs = np.stack([d[0] for d in data]).astype(np.uint8)
    msks = np.stack([d[1] for d in data]).astype(np.uint8)
    if USE_CACHE:
        np.savez_compressed(cache, imgs=imgs, msks=msks)
        print(f"[{name}] saved cache: {os.path.getsize(cache)/1024**2:.1f} MB "
              f"(raw uint8 {(imgs.nbytes + msks.nbytes)/1024**2:.1f} MB)")
    return imgs, msks


def normalize(img):
    return (img.astype(np.float32) / 255.0 - MEAN) / STD

# [COMPRESS] augmentation in pure TF ops (no cv2 / numpy_function -> no Python GIL bottleneck)
def tf_augment(img, msk):
    """img: uint8 (H,W,3), msk: uint8 (H,W). Returns normalized float32 image and float32 mask."""
    s = IMG_SIZE
    img = tf.cast(img, tf.float32); msk = tf.cast(msk, tf.float32)[..., None]
    # flips
    if_h = tf.random.uniform(()) < 0.5
    img, msk = tf.cond(if_h, lambda: (img[:, ::-1], msk[:, ::-1]), lambda: (img, msk))
    if_v = tf.random.uniform(()) < 0.5
    img, msk = tf.cond(if_v, lambda: (img[::-1], msk[::-1]), lambda: (img, msk))
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img, msk = tf.image.rot90(img, k), tf.image.rot90(msk, k)
    # random affine (rotation +-30deg, scale 0.85-1.2, shift +-10%), applied with prob 0.7
    ang = tf.random.uniform((), -30., 30.) * math.pi / 180.0
    sc  = tf.random.uniform((), 0.85, 1.2)
    tx  = tf.random.uniform((), -0.1, 0.1) * s
    ty  = tf.random.uniform((), -0.1, 0.1) * s
    c, sn, c0 = tf.cos(ang) / sc, tf.sin(ang) / sc, s / 2.0
    T = tf.stack([c, -sn, c0 - c * c0 + sn * c0 + tx,
                  sn,  c, c0 - sn * c0 - c * c0 + ty, 0., 0.])
    ident = tf.constant([1., 0., 0., 0., 1., 0., 0., 0.])
    T = tf.where(tf.random.uniform(()) < 0.7, T, ident)[None]
    def warp(t, mode):
        return tf.raw_ops.ImageProjectiveTransformV3(
            images=t[None], transforms=T, output_shape=[s, s], fill_value=0.0,
            interpolation=mode, fill_mode="REFLECT")[0]
    img, msk = warp(img, "BILINEAR"), warp(msk, "NEAREST")
    # brightness / contrast (prob 0.7)
    bc = tf.clip_by_value(img * tf.random.uniform((), 0.8, 1.2) + tf.random.uniform((), -20., 20.), 0., 255.)
    img = tf.where(tf.random.uniform(()) < 0.7, bc, img)
    # hue / saturation jitter (prob 0.3)
    hs = tf.image.random_saturation(tf.image.random_hue(img / 255.0, 0.04), 0.9, 1.1) * 255.0
    img = tf.where(tf.random.uniform(()) < 0.3, tf.clip_by_value(hs, 0., 255.), img)
    # normalize last, so stored data stays uint8
    img = (img / 255.0 - MEAN) / STD
    return img, tf.cast(msk > 0.5, tf.float32)

def make_train_ds(imgs, msks):
    n = len(imgs)
    ds = tf.data.Dataset.from_tensor_slices((imgs, msks))          # uint8 tensors (compressed in memory)
    return (ds.shuffle(n, reshuffle_each_iteration=True)
              .map(tf_augment, num_parallel_calls=AUTOTUNE)
              .batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTOTUNE))

def prep_eval(imgs, msks):
    x = np.stack([normalize(i) for i in imgs])
    y = msks.astype(np.float32)[..., None]
    return x, y


# --------------------------- LOSS / METRICS --------------------------
def seg_loss(y_true, logits):
    y_true = tf.cast(y_true, tf.float32); logits = tf.cast(logits, tf.float32)
    bce = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y_true, logits=logits))
    p = tf.reshape(tf.sigmoid(logits), (tf.shape(logits)[0], -1))
    g = tf.reshape(y_true, (tf.shape(logits)[0], -1))
    d = 1.0 - (2 * tf.reduce_sum(p * g, 1) + 1.0) / (tf.reduce_sum(p, 1) + tf.reduce_sum(g, 1) + 1.0)
    return bce + tf.reduce_mean(d)


def _bin(logits):
    return tf.cast(tf.sigmoid(tf.cast(logits, tf.float32)) > 0.5, tf.float32)


def dice(y_true, y_pred):
    p, g = _bin(y_pred), tf.cast(y_true, tf.float32)
    tp = tf.reduce_sum(p * g, (1, 2, 3))
    s = tf.reduce_sum(p, (1, 2, 3)) + tf.reduce_sum(g, (1, 2, 3))
    return tf.reduce_mean((2 * tp + 1e-7) / (s + 1e-7))


def iou(y_true, y_pred):
    p, g = _bin(y_pred), tf.cast(y_true, tf.float32)
    tp = tf.reduce_sum(p * g, (1, 2, 3))
    u = tf.reduce_sum(p, (1, 2, 3)) + tf.reduce_sum(g, (1, 2, 3)) - tp
    return tf.reduce_mean((tp + 1e-7) / (u + 1e-7))


def batch_metrics(logits, gt, thr=0.5, eps=1e-7):
    logits = np.clip(logits.astype(np.float64), -50, 50)
    p = (1 / (1 + np.exp(-logits)) > thr).astype(np.float64).reshape(len(gt), -1)
    g = gt.astype(np.float64).reshape(len(gt), -1)
    tp = (p * g).sum(1); fp = (p * (1 - g)).sum(1); fn = ((1 - p) * g).sum(1); tn = ((1 - p) * (1 - g)).sum(1)
    return {
        "dice":        (2 * tp + eps) / (2 * tp + fp + fn + eps),
        "iou":         (tp + eps) / (tp + fp + fn + eps),
        "precision":   (tp + eps) / (tp + fp + eps),
        "recall":      (tp + eps) / (tp + fn + eps),
        "specificity": (tn + eps) / (tn + fp + eps),
        "accuracy":    (tp + tn) / (tp + tn + fp + fn),
    }


def evaluate(model, x, y):
    logits = model.predict(x, batch_size=32, verbose=0)
    res = {k: float(v.mean()) for k, v in batch_metrics(logits, y).items()}
    res["loss"] = float(seg_loss(y, logits).numpy())
    return res


# ------------------------------ TRAINING -----------------------------
class OneCycle(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, max_lr, total_steps, pct=0.1, div=25.0, final=1e-6):
        super().__init__()
        self.max_lr, self.total, self.pct, self.div, self.final = max_lr, total_steps, pct, div, final

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = float(self.total) * self.pct
        start = self.max_lr / self.div
        lr_w = start + (self.max_lr - start) * step / warm
        prog = tf.clip_by_value((step - warm) / (float(self.total) - warm), 0.0, 1.0)
        lr_c = self.final + (self.max_lr - self.final) * 0.5 * (1.0 + tf.cos(math.pi * prog))
        return tf.where(step < warm, lr_w, lr_c)

    def get_config(self):
        return dict(max_lr=self.max_lr, total_steps=self.total, pct=self.pct, div=self.div, final=self.final)


class TrainMonitor(tf.keras.callbacks.Callback):
    """Progress %, ETA, all metrics every epoch, CSV log, best-model (.keras) saving, early stopping."""
    def __init__(self, xv, yv, steps_per_epoch, sched):
        super().__init__()
        self.xv, self.yv, self.spe, self.sched = xv, yv, steps_per_epoch, sched
        self.best, self.bad = -1.0, 0
        self.export = build_model()
        self.csv_path = BASE + "_log.csv"

    def on_train_begin(self, logs=None):
        self.t_start = time.time()
        with open(self.csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["epoch", "lr", "train_loss", "train_dice", "train_iou", "val_loss"]
                                   + [f"val_{k}" for k in METRIC_KEYS] + ["best_val_dice"])

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()
        pct = 100.0 * epoch / EPOCHS
        self.pbar = tqdm(total=self.spe, desc=f"Ep {epoch+1:03d}/{EPOCHS} [{pct:5.1f}%]",
                         leave=False, ncols=100)

    def on_train_batch_end(self, batch, logs=None):
        self.pbar.update(1)
        if logs:
            self.pbar.set_postfix(loss=f"{logs.get('loss', 0.0):.4f}")

    def on_epoch_end(self, epoch, logs=None):
        self.pbar.close()
        logs = logs or {}
        v = evaluate(self.model, self.xv, self.yv)
        done = epoch + 1
        pct = 100.0 * done / EPOCHS
        elapsed = time.time() - self.t_start
        eta = elapsed / done * (EPOCHS - done)
        lr = float(self.sched(tf.constant(done * self.spe, dtype=tf.float32)))

        improved = v["dice"] > self.best
        if improved:
            self.best, self.bad = v["dice"], 0
            self.export.set_weights(self.model.get_weights())
            self.export.save(SAVE_PATH)
        else:
            self.bad += 1

        print(f"[Ep {done:03d}/{EPOCHS} | {pct:5.1f}%] {time.time()-self.t0:5.1f}s | "
              f"ETA {eta/60:6.1f} min | lr {lr:.2e}")
        print(f"   train: loss {logs.get('loss', float('nan')):.4f} | "
              f"dice {logs.get('dice', float('nan')):.4f} | iou {logs.get('iou', float('nan')):.4f}")
        print(f"   val  : loss {v['loss']:.4f} | dice {v['dice']:.4f} | iou {v['iou']:.4f} | "
              f"prec {v['precision']:.4f} | rec {v['recall']:.4f} | "
              f"spec {v['specificity']:.4f} | acc {v['accuracy']:.4f}")
        print(f"   best val dice {self.best:.4f}" + ("  <-- new best, model saved" if improved
                                                      else f"  (no gain for {self.bad}/{PATIENCE})"))

        with open(self.csv_path, "a", newline="") as f:
            csv.writer(f).writerow([done, lr, logs.get("loss"), logs.get("dice"), logs.get("iou"), v["loss"]]
                                   + [v[k] for k in METRIC_KEYS] + [self.best])

        if self.bad >= PATIENCE:
            print("Early stopping.")
            self.model.stop_training = True


# ------------------------------- FPS ---------------------------------
def _make_infer(model):
    @tf.function
    def infer(x):
        return tf.sigmoid(model(x, training=False))
    return infer


def measure_fps(model, batch=1, warmup=50, runs=300):
    infer = _make_infer(model)
    x = tf.constant(np.random.rand(batch, IMG_SIZE, IMG_SIZE, 3).astype(np.float32))
    for _ in range(warmup):
        infer(x).numpy()
    t0 = time.perf_counter()
    for _ in range(runs):
        infer(x).numpy()
    return batch * runs / (time.perf_counter() - t0)

def measure_fps_e2e(model, img_path, warmup=30, runs=200):
    """Full pipeline: resize + normalize + inference + threshold (on a real image)."""
    raw = cv2.imread(img_path)
    infer = _make_infer(model)

    def step():
        im = cv2.cvtColor(cv2.resize(raw, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
        x = tf.constant(normalize(im)[None])
        return infer(x).numpy()[0, ..., 0] > 0.5

    for _ in range(warmup):
        step()
    t0 = time.perf_counter()
    for _ in range(runs):
        step()
    return runs / (time.perf_counter() - t0)

# ------------------------------- MAIN --------------------------------
def train():
    seed_everything(SEED)
    os.makedirs(os.path.dirname(os.path.abspath(SAVE_PATH)), exist_ok=True)

    # let the GPU allocate memory gradually instead of grabbing all 8 GB at once
    for g in tf.config.list_physical_devices("GPU"):
        try: tf.config.experimental.set_memory_growth(g, True)
        except Exception: pass

    tr, va, te = build_splits(IMG_DIR, MASK_DIR)
    print(f"Split -> train {len(tr)} | val {len(va)} | test {len(te)}")
    with open(BASE + "_splits.json", "w") as f:
        json.dump({"train": tr, "val": va, "test": te}, f, indent=1)

    # [COMPRESS] compressed, cached uint8 arrays for every split
    tr_i, tr_m = compress_dataset(tr, "train")
    va_i, va_m = compress_dataset(va, "val")
    te_i, te_m = compress_dataset(te, "test")
    train_ds = make_train_ds(tr_i, tr_m)
    xv, yv = prep_eval(va_i, va_m)
    xt, yt = prep_eval(te_i, te_m)

    gpu = len(tf.config.list_physical_devices("GPU")) > 0
    print("Device:", "GPU" if gpu else "CPU (no GPU detected)")
    tf.keras.mixed_precision.set_global_policy("mixed_float16" if (gpu and USE_MIXED_PRECISION) else "float32")

    model = build_model()
    print(f"Params: {count_params(model):.4f} M | GMACs: {count_gmacs(model):.3f}")

    spe = len(tr_i) // BATCH_SIZE
    sched = OneCycle(MAX_LR, spe * EPOCHS)
    opt = tf.keras.optimizers.AdamW(sched, weight_decay=WEIGHT_DECAY, global_clipnorm=5.0)
    model.compile(optimizer=opt, loss=seg_loss, metrics=[dice, iou], jit_compile=False)
    model.fit(train_ds, epochs=EPOCHS, verbose=0, callbacks=[TrainMonitor(xv, yv, spe, sched)])

    # ---- reload the best saved .keras model and test it
    model = tf.keras.models.load_model(SAVE_PATH, compile=False)
    res = evaluate(model, xt, yt)
    params, gmacs = count_params(model), count_gmacs(model)
    size_mb = os.path.getsize(SAVE_PATH) / 1024 ** 2
    fps = measure_fps(model)
    fps_e2e = measure_fps_e2e(model, te[0][0])

    print("\n================ TEST RESULTS ================")
    rows = [("GMACs", gmacs, f"< {TARGETS['gmacs']}", gmacs < TARGETS["gmacs"]),
            ("Params (M)", params, f"<= {TARGETS['params_m']}", params <= TARGETS["params_m"]),
            ("Size (MB)", size_mb, f"< {TARGETS['size_mb']}", size_mb < TARGETS["size_mb"]),
            ("FPS (model)", fps, f">= {TARGETS['fps']:.0f}", fps >= TARGETS["fps"])]
    for k in METRIC_KEYS:
        rows.append((k, res[k], f"> {TARGETS['metric']}", res[k] > TARGETS["metric"]))
    for name, val, tgt, ok in rows:
        print(f"{name:<12} {val:10.4f}   target {tgt:<8}  {'PASS' if ok else 'FAIL'}")
    print(f"{'FPS (e2e)':<12} {fps_e2e:10.4f}   (resize + normalize + model + threshold)")
    out = {r[0]: float(r[1]) for r in rows}
    out["FPS (e2e)"] = float(fps_e2e)
    with open(BASE + "_test_metrics.json", "w") as f:
        json.dump(out, f, indent=1)

    explain(model, xt, yt, os.path.join(os.path.dirname(os.path.abspath(SAVE_PATH)), "xai"))

# --------------------------- EXPLAINABILITY --------------------------
def _norm(c):
    c = c - c.min()
    return c / (c.max() + 1e-8)
def _resize(t):
    t = t if t.shape.rank == 4 else t[None]
    return tf.image.resize(t, (IMG_SIZE, IMG_SIZE), method="bilinear")[0]
def grad_cam(model, x, layer_name):
    gm = Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        act, logits = gm(x, training=False)
        mask = tf.cast(tf.sigmoid(logits) > 0.5, tf.float32)
        if tf.reduce_sum(mask) == 0: mask = tf.ones_like(mask)
        target = tf.reduce_sum(logits * mask)
    grads = tape.gradient(target, act)
    w = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * act, -1, keepdims=True))
    return _norm(_resize(cam).numpy()[..., 0])
def score_cam(model, x, layer_name, chunk=32):
    gm = Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    act, logits = gm(x, training=False)
    mask = tf.cast(tf.sigmoid(logits) > 0.5, tf.float32)
    if tf.reduce_sum(mask) == 0: mask = tf.ones_like(mask)
    up = _resize(act).numpy()
    up = (up - up.min((0, 1), keepdims=True)) / (up.max((0, 1), keepdims=True) - up.min((0, 1), keepdims=True) + 1e-8)
    up = tf.constant(np.transpose(up, (2, 0, 1))[..., None].astype(np.float32))
    scores = []
    for i in range(0, up.shape[0], chunk):
        p = tf.sigmoid(model(x * up[i:i + chunk], training=False))
        scores.append(tf.reduce_sum(p * mask, (1, 2, 3)) / tf.reduce_sum(mask))
    w = tf.nn.softmax(tf.concat(scores, 0))
    cam = tf.nn.relu(tf.reduce_sum(w * act[0], -1, keepdims=True))
    return _norm(_resize(cam).numpy()[..., 0])
def denorm(x):
    return np.clip(x * STD + MEAN, 0, 1)
def overlay(img, cam):
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1] / 255.0
    return np.clip(0.55 * img + 0.45 * heat, 0, 1)

def save_feature_maps(model, x, path, n=16, names=("stem_out", "enc2_out", "enc3_out", "dec3_out")):
    fm = Model(model.inputs, [model.get_layer(k).output for k in names])
    feats = [f[0].numpy() for f in fm(x, training=False)]
    fig, axes = plt.subplots(len(names), n, figsize=(n * 1.1, len(names) * 1.2))
    for r, (k, f) in enumerate(zip(names, feats)):
        for c in range(n):
            ax = axes[r, c]; ax.imshow(f[..., c], cmap="viridis"); ax.axis("off")
            if c == 0: ax.set_title(k, fontsize=7, loc="left")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def explain(model, xs, ys, out_dir, layer_name="dec3_out"):
    os.makedirs(out_dir, exist_ok=True)
    idx = np.linspace(0, len(xs) - 1, min(N_XAI_IMAGES, len(xs))).astype(int)
    for j, i in enumerate(idx):
        x = xs[i][None]
        pred = (tf.sigmoid(model(x, training=False)).numpy()[0, ..., 0] > 0.5)
        img = denorm(xs[i])
        g, s = grad_cam(model, x, layer_name), score_cam(model, x, layer_name)
        fig, ax = plt.subplots(1, 5, figsize=(13, 2.8))
        for a, im, t in zip(ax, [img, ys[i][..., 0], pred, overlay(img, g), overlay(img, s)],
                            ["Image", "Ground truth", "Prediction", "Grad-CAM", "Score-CAM"]):
            a.imshow(im, cmap="gray" if im.ndim == 2 else None); a.set_title(t, fontsize=9); a.axis("off")
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"cam_{j:02d}.png"), dpi=150); plt.close()
        if j == 0:
            save_feature_maps(model, x, os.path.join(out_dir, "feature_maps.png"))
    print(f"XAI figures saved to: {out_dir}")
